# Local LLM Positional Bias Evaluation

This notebook evaluates positional bias in Large Language Models (LLMs) using multiple choice questions via Ollama.

**Positional bias** refers to the tendency of models to prefer certain answer positions (A, B, C, D) over others, regardless of content.

## Methodology
1. Load multiple choice questions from CSV
2. For each question, create multiple permutations of the answer options
3. Test the model on each permutation
4. Analyze the distribution of chosen answers to detect bias

## Requirements
- Ollama running locally
- Model downloaded in Ollama
- Required Python packages (see requirements.txt)

## 1. Import Required Libraries

In [ ]:
import csv
import json
import re
import time
import os
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from scipy.stats import chisquare
from tqdm import tqdm
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

## 2. Configuration and Parameters

Set your evaluation parameters here. Modify these values to test different models or configurations.

In [ ]:
# =================================
# CONFIGURATION - Modify as needed
# =================================

# Model and connection settings
MODEL_NAME = "qwen2.5:14b-instruct-q8_0"  # Change this to your desired model
OLLAMA_HOST = "http://localhost:11434"

# Evaluation parameters
CSV_PATH = "ict_pp/csv/2012-2020.csv"  # Path to your MCQ dataset
N_PERMUTATIONS = 50  # Number of permutations per question (4 is minimum for full coverage)
MAX_QUESTIONS = 20   # Limit questions for testing (set to None for all questions)
TEMPERATURE = 0.0    # Model temperature (0.0 = deterministic)
SEED = 42           # Random seed for reproducibility

# Output settings
OUTPUT_PREFIX = "results/positional_bias"

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Host: {OLLAMA_HOST}")
print(f"  Dataset: {CSV_PATH}")
print(f"  Permutations per question: {N_PERMUTATIONS}")
print(f"  Max questions: {MAX_QUESTIONS if MAX_QUESTIONS else 'All'}")
print(f"  Temperature: {TEMPERATURE}")

## 3. Data Structures and Prompt Template

In [ ]:
# Prompt template for multiple choice questions
PROMPT_TEMPLATE = """Question: {question}

A. {A}
B. {B}
C. {C}
D. {D}

You must respond with exactly one letter (A, B, C, or D) and nothing else.
Answer:"""

# Regex to extract answer letter
LETTER_RE = re.compile(r'\b([A-D])\b')

@dataclass
class MCQ:
    uid: str
    question: str
    options: Dict[str, str]  # keys A-D, values are option text
    answer: str  # correct letter A-D

print("✓ Data structures and prompt template defined!")
print(f"\nPrompt template preview:")
print(PROMPT_TEMPLATE.format(
    question="Sample question?",
    A="Option A", B="Option B", C="Option C", D="Option D"
))

## 4. Data Loading Functions

In [ ]:
def load_mcq_csv(path: str, max_questions: int = None) -> List[MCQ]:
    """Load multiple choice questions from CSV file"""
    df = pd.read_csv(path)
    required_cols = {"id", "question", "option_a", "option_b", "option_c", "option_d", "answer"}
                
    if not required_cols.issubset(df.columns):
        missing = required_cols - set(df.columns)
        raise ValueError(f"Missing columns in CSV: {missing}")

    mcqs = []
    for _, row in df.iterrows():
        ans = str(row["answer"]).strip().upper()
        if ans not in {"A", "B", "C", "D"}:
            print(f"Skipping question {row['id']} - invalid answer: {ans}")
            continue
        
        # Check for missing question or options
        question = str(row["question"]).strip()
        options_text = [
            str(row["option_a"]).strip(),
            str(row["option_b"]).strip(), 
            str(row["option_c"]).strip(),
            str(row["option_d"]).strip()
        ]
        
        if (question in ["", "nan"] or 
            any(opt in ["", "nan"] for opt in options_text)):
            print(f"Skipping question {row['id']} - incomplete data")
            continue
            
        mcq = MCQ(
            uid=str(row["id"]),
            question=question,
            options={
                "A": options_text[0],
                "B": options_text[1],
                "C": options_text[2],
                "D": options_text[3],
            },
            answer=ans,
        )
        mcqs.append(mcq)
    
    if max_questions:
        mcqs = mcqs[:max_questions]
    
    print(f"Loaded {len(mcqs)} questions from {path}")
    return mcqs

# Test the loading function
print("Testing data loading...")
test_mcqs = load_mcq_csv(CSV_PATH, max_questions=3)
print(f"\nSample question:")
if test_mcqs:
    sample = test_mcqs[0]
    print(f"ID: {sample.uid}")
    print(f"Question: {sample.question[:100]}...")
    print(f"Correct answer: {sample.answer}")
    for letter, option in sample.options.items():
        print(f"  {letter}: {option[:60]}...")

## 5. Option Permutation Functions

In [ ]:
def permute_options(options: Dict[str, str], perm_idx: int) -> Tuple[Dict[str, str], Dict[str, str]]:
    """
    Cyclically permute the options and return the new mapping
    Returns: (new_options, mapping from new_letter -> original_letter)
    """
    letters = ["A", "B", "C", "D"]
    original_items = [(letter, options[letter]) for letter in letters]
    
    # Cycle through permutations: shift by (perm_idx % 4)
    shift = perm_idx % 4
    cyclic_items = original_items[shift:] + original_items[:shift]
    
    # Create new mapping
    new_options = {letters[i]: cyclic_items[i][1] for i in range(4)}
    # Track which new position corresponds to which original position
    new_to_old_mapping = {letters[i]: cyclic_items[i][0] for i in range(4)}
    
    return new_options, new_to_old_mapping

def build_prompt(mcq: MCQ, permuted_options: Dict[str, str]) -> str:
    """Build the full prompt for the LLM"""
    return PROMPT_TEMPLATE.format(
        question=mcq.question,
        A=permuted_options["A"],
        B=permuted_options["B"],
        C=permuted_options["C"],
        D=permuted_options["D"],
    )

# Test permutation function
print("Testing option permutation...")
if test_mcqs:
    sample = test_mcqs[0]
    print(f"\nOriginal options:")
    for letter, option in sample.options.items():
        print(f"  {letter}: {option[:50]}...")
    
    # Test a few permutations
    for i in range(3):
        perm_options, mapping = permute_options(sample.options, i)
        print(f"\nPermutation {i}:")
        for letter, option in perm_options.items():
            original = mapping[letter]
            print(f"  {letter}: {option[:50]}... (was {original})")

## 6. Ollama Communication Functions

In [ ]:
def call_ollama(model: str, prompt: str, host: str = "http://localhost:11434", 
                temperature: float = 0.0, seed: int = 42, retries: int = 3, 
                timeout: int = 60) -> str:
    """Call Ollama API to get model response"""
    url = f"{host.rstrip('/')}/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 1.0,
            "seed": seed,
            "num_ctx": 4096,
            "stop": ["Question:", "Options:", "\n\nQuestion:"]
        }
    }
    
    for attempt in range(retries):
        try:
            response = requests.post(url, json=payload, timeout=timeout)
            response.raise_for_status()
            
            data = response.json()
            text = data.get("response", "").strip()
            
            if not text:
                raise RuntimeError("Empty response from model")
                
            return text
            
        except requests.exceptions.RequestException as e:
            print(f"Request error (attempt {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                raise
            time.sleep(1.0 + attempt * 0.5)
        except Exception as e:
            print(f"Unexpected error (attempt {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                raise
            time.sleep(1.0)
    
    return ""

# Test Ollama connection
print("Testing Ollama connection...")
try:
    test_response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    test_response.raise_for_status()
    print(f"✓ Successfully connected to Ollama at {OLLAMA_HOST}")
    
    # List available models
    models_data = test_response.json()
    available_models = [model['name'] for model in models_data.get('models', [])]
    print(f"Available models: {available_models}")
    
    if MODEL_NAME in available_models:
        print(f"✓ Target model '{MODEL_NAME}' is available")
    else:
        print(f"⚠️  Target model '{MODEL_NAME}' not found. Available models: {available_models}")
    
except Exception as e:
    print(f"❌ Failed to connect to Ollama: {e}")
    print("Make sure Ollama is running and accessible")

## 7. Answer Parsing Functions

In [ ]:
def parse_answer(response_text: str) -> str:
    """Extract the answer letter from model response"""
    response_text = response_text.strip().upper()
    
    # Check if response contains think tags (even incomplete ones)
    if "<THINK>" in response_text:
        # Extract text from opening think tag to end (handle incomplete responses)
        think_start = response_text.find("<THINK>")
        if "</THINK>" in response_text:
            think_end = response_text.find("</THINK>")
            think_content = response_text[think_start:think_end]
        else:
            # Handle incomplete think blocks
            think_content = response_text[think_start:]
        
        # Enhanced answer patterns for qwen model's reasoning style
        answer_patterns = [
            r'ANSWER\s+(?:SHOULD\s+BE|IS|MUST\s+BE)\s+([A-D])',
            r'SO\s+(?:THE\s+)?ANSWER\s+(?:SHOULD\s+BE|IS|MUST\s+BE)\s+([A-D])',
            r'(?:DEFINITELY|CLEARLY)\s+([A-D])',
            r'(?:SO|THEREFORE),?\s+(?:THE\s+ANSWER\s+IS\s+)?([A-D])',
            r'OPTION\s+([A-D])',
            r'CHOICE\s+([A-D])',
            r'([A-D]),?\s+(?:IS\s+THE\s+ANSWER|IS\s+CORRECT)'
        ]
        
        for pattern in answer_patterns:
            match = re.search(pattern, think_content)
            if match:
                return match.group(1)
    
    # Enhanced fallback: look for letters in context
    # Prioritize letters that appear after reasoning words
    reasoning_patterns = [
        r'(?:ANSWER|OPTION|CHOICE|SO|THEREFORE)\s+(?:IS\s+)?([A-D])',
        r'([A-D])\s+(?:IS\s+(?:THE\s+)?(?:ANSWER|CORRECT|RIGHT))',
        r'MUST\s+BE\s+([A-D])'
    ]
    
    for pattern in reasoning_patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return matches[-1]  # Return the last match
    
    # Final fallback: look for standalone letters (but prefer later ones)
    matches = re.findall(r'\b([A-D])\b', response_text)
    if matches:
        return matches[-1]  # Return the last mentioned letter
    
    return ""

# Test answer parsing with sample responses
print("Testing answer parsing...")
test_responses = [
    "A",
    "The answer is B.",
    "Looking at the options, I think C is correct.",
    "After considering all factors, the answer must be D.",
    "This is tricky but I'll go with A"
]

for i, response in enumerate(test_responses, 1):
    parsed = parse_answer(response)
    print(f"Test {i}: '{response}' -> '{parsed}'")

## 8. Test Model Response (Optional)

Let's test the model with a sample question to make sure everything is working:

In [ ]:
# Optional: Test with a sample question
print("Testing model response with sample question...")

if test_mcqs:
    sample_mcq = test_mcqs[0]
    sample_prompt = build_prompt(sample_mcq, sample_mcq.options)
    
    print(f"Sample prompt:")
    print("-" * 50)
    print(sample_prompt)
    print("-" * 50)
    
    try:
        response = call_ollama(
            model=MODEL_NAME,
            prompt=sample_prompt,
            host=OLLAMA_HOST,
            temperature=TEMPERATURE,
            seed=SEED,
            timeout=30
        )
        
        parsed_answer = parse_answer(response)
        
        print(f"\nModel response:")
        print(f"Raw: '{response}'")
        print(f"Parsed answer: '{parsed_answer}'")
        print(f"Correct answer: '{sample_mcq.answer}'")
        print(f"Is correct: {parsed_answer == sample_mcq.answer}")
        
    except Exception as e:
        print(f"❌ Error testing model: {e}")
        print("Please check that the model is available and Ollama is running")
else:
    print("No test questions available")

## 9. Main Evaluation Function

In [ ]:
def run_evaluation(model: str, host: str, csv_path: str, n_permutations: int,
                  temperature: float, seed: int, max_questions: int,
                  output_prefix: str):
    """Run the full positional bias evaluation"""
    
    print(f"\n=== Starting Positional Bias Evaluation ===")
    print(f"Model: {model}")
    print(f"Host: {host}")
    print(f"Dataset: {csv_path}")
    print(f"Permutations per question: {n_permutations}")
    print(f"Temperature: {temperature}")
    print(f"Seed: {seed}")
    
    # Load questions
    mcqs = load_mcq_csv(csv_path, max_questions=max_questions)
    
    # Test Ollama connection
    try:
        test_response = requests.get(f"{host}/api/tags", timeout=10)
        test_response.raise_for_status()
        print(f"✓ Successfully connected to Ollama at {host}")
    except Exception as e:
        print(f"❌ Failed to connect to Ollama: {e}")
        print("Make sure Ollama is running and accessible")
        return None
    
    # Run evaluation
    results = []
    total_prompts = len(mcqs) * n_permutations
    
    print(f"\nProcessing {len(mcqs)} questions with {n_permutations} permutations each...")
    print(f"Total prompts: {total_prompts}")
    
    with tqdm(total=total_prompts, desc=f"Evaluating {model}") as pbar:
        for mcq in mcqs:
            for perm_idx in range(n_permutations):
                # Create permuted version of options
                permuted_options, new_to_old_mapping = permute_options(mcq.options, perm_idx)
                
                # Find where the correct answer ended up
                correct_new_position = None
                for new_pos, old_pos in new_to_old_mapping.items():
                    if old_pos == mcq.answer:
                        correct_new_position = new_pos
                        break
                
                # Build prompt and get model response
                prompt = build_prompt(mcq, permuted_options)
                
                try:
                    response_text = call_ollama(
                        model=model,
                        prompt=prompt,
                        host=host,
                        temperature=temperature,
                        seed=seed + perm_idx,
                        timeout=120
                    )
                    
                    predicted_answer = parse_answer(response_text)
                    is_correct = (predicted_answer == correct_new_position)
                    
                except Exception as e:
                    print(f"\nError processing {mcq.uid} perm {perm_idx}: {e}")
                    response_text = ""
                    predicted_answer = ""
                    is_correct = False
                
                # Store result
                results.append({
                    "question_id": mcq.uid,
                    "permutation_idx": perm_idx,
                    "model": model,
                    "predicted_answer": predicted_answer,
                    "correct_position": correct_new_position,
                    "original_correct": mcq.answer,
                    "is_correct": int(is_correct),
                    "raw_response": response_text.replace('\n', ' ').replace('\r', ''),
                    "question": mcq.question,
                    "option_A": permuted_options["A"],
                    "option_B": permuted_options["B"],
                    "option_C": permuted_options["C"],
                    "option_D": permuted_options["D"],
                })
                
                pbar.update(1)
    
    # Create results DataFrame
    df = pd.DataFrame(results)
    
    # Save results to CSV
    os.makedirs(os.path.dirname(output_prefix) if os.path.dirname(output_prefix) else ".", exist_ok=True)
    output_file = f"{output_prefix}_{model.replace(':', '_').replace('/', '_')}.csv"
    df.to_csv(output_file, index=False)
    
    print(f"\n✓ Results saved to: {output_file}")
    
    return df, output_file

print("✓ Evaluation function defined!")

## 10. Results Analysis Functions

In [ ]:
def analyze_results(df: pd.DataFrame, model: str, output_file: str):
    """Analyze and print results of positional bias evaluation"""
    
    print(f"\n=== POSITIONAL BIAS ANALYSIS for {model} ===")
    
    # Filter out failed responses
    valid_responses = df[df["predicted_answer"].isin(["A", "B", "C", "D"])]
    failed_responses = len(df) - len(valid_responses)
    
    if failed_responses > 0:
        print(f"⚠️  WARNING: {failed_responses}/{len(df)} responses failed to parse")
    
    if len(valid_responses) == 0:
        print("❌ ERROR: No valid responses to analyze")
        return
    
    # Overall choice distribution
    choice_counts = valid_responses["predicted_answer"].value_counts().reindex(["A", "B", "C", "D"], fill_value=0)
    total_valid = len(valid_responses)
    
    print(f"\n📊 CHOICE DISTRIBUTION (n={total_valid}):")
    for letter in ["A", "B", "C", "D"]:
        count = choice_counts[letter]
        percentage = (count / total_valid * 100) if total_valid > 0 else 0
        print(f"   {letter}: {count:4d} ({percentage:5.1f}%)")
    
    # Chi-square test against uniform distribution
    expected_per_choice = total_valid / 4
    expected = [expected_per_choice] * 4
    chi2_stat, p_value = chisquare(choice_counts.values, f_exp=expected)
    
    print(f"\n🔬 CHI-SQUARE TEST vs Uniform Distribution:")
    print(f"   Chi-square statistic: {chi2_stat:.3f}")
    print(f"   P-value: {p_value:.6f}")
    if p_value < 0.05:
        print("   🚨 Significant deviation from uniform (p < 0.05) - BIAS DETECTED")
    else:
        print("   ✅ No significant deviation from uniform (p >= 0.05)")
    
    # Accuracy by position of correct answer
    print(f"\n🎯 ACCURACY BY CORRECT ANSWER POSITION:")
    accuracy_by_position = valid_responses.groupby("correct_position")["is_correct"].agg(['mean', 'count'])
    
    overall_accuracy = valid_responses["is_correct"].mean()
    print(f"   Overall accuracy: {overall_accuracy:.3f} ({overall_accuracy*100:.1f}%)")
    print(f"   Position-specific accuracy:")
    
    for letter in ["A", "B", "C", "D"]:
        if letter in accuracy_by_position.index:
            acc = accuracy_by_position.loc[letter, "mean"]
            count = accuracy_by_position.loc[letter, "count"]
            diff = acc - overall_accuracy
            print(f"     {letter}: {acc:.3f} ({acc*100:.1f}%) [n={count}, diff={diff:+.3f}]")
        else:
            print(f"     {letter}: N/A (no questions)")
    
    # Position bias score (standard deviation of choice percentages)
    choice_percentages = choice_counts.values / total_valid * 100
    position_bias_score = np.std(choice_percentages)
    print(f"\n📈 POSITION BIAS SCORE: {position_bias_score:.2f}")
    print(f"   (Standard deviation of choice percentages - higher = more biased)")
    
    print(f"\n💾 Full results saved to: {output_file}")
    
    # Summary
    if p_value < 0.05 or position_bias_score > 5:
        print(f"\n🔍 CONCLUSION: {model} shows evidence of positional bias")
    else:
        print(f"\n✅ CONCLUSION: {model} shows minimal positional bias")
    
    return {
        'choice_counts': choice_counts,
        'choice_percentages': choice_percentages,
        'chi2_stat': chi2_stat,
        'p_value': p_value,
        'overall_accuracy': overall_accuracy,
        'accuracy_by_position': accuracy_by_position,
        'position_bias_score': position_bias_score,
        'total_responses': total_valid,
        'failed_responses': failed_responses
    }

print("✓ Analysis function defined!")

## 11. Visualization Functions

In [ ]:
def create_visualizations(analysis_results: dict, model: str):
    """Create visualizations for the bias analysis"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Positional Bias Analysis: {model}', fontsize=16, fontweight='bold')
    
    # 1. Choice distribution bar chart
    letters = ['A', 'B', 'C', 'D']
    counts = analysis_results['choice_counts'].values
    percentages = analysis_results['choice_percentages']
    
    bars = ax1.bar(letters, percentages, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
    ax1.axhline(y=25, color='red', linestyle='--', alpha=0.7, label='Expected (25%)')
    ax1.set_title('Choice Distribution')
    ax1.set_ylabel('Percentage (%)')
    ax1.set_ylim(0, max(percentages) * 1.1)
    ax1.legend()
    
    # Add value labels on bars
    for bar, pct in zip(bars, percentages):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom')
    
    # 2. Accuracy by position
    if 'accuracy_by_position' in analysis_results:
        acc_data = analysis_results['accuracy_by_position']
        acc_letters = []
        acc_values = []
        for letter in letters:
            if letter in acc_data.index:
                acc_letters.append(letter)
                acc_values.append(acc_data.loc[letter, 'mean'])
        
        bars2 = ax2.bar(acc_letters, [v*100 for v in acc_values], 
                       color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'][:len(acc_letters)])
        overall_acc = analysis_results['overall_accuracy'] * 100
        ax2.axhline(y=overall_acc, color='red', linestyle='--', alpha=0.7, 
                   label=f'Overall ({overall_acc:.1f}%)')
        ax2.set_title('Accuracy by Correct Answer Position')
        ax2.set_ylabel('Accuracy (%)')
        ax2.set_ylim(0, 100)
        ax2.legend()
        
        # Add value labels
        for bar, acc in zip(bars2, acc_values):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{acc*100:.1f}%', ha='center', va='bottom')
    
    # 3. Bias metrics
    metrics = ['Chi-square', 'P-value', 'Bias Score']
    values = [analysis_results['chi2_stat'], 
              analysis_results['p_value'] * 100,  # Scale p-value for visibility
              analysis_results['position_bias_score']]
    
    bars3 = ax3.bar(metrics, values, color=['#96CEB4', '#FFEAA7', '#DDA0DD'])
    ax3.set_title('Bias Metrics')
    ax3.set_ylabel('Value')
    
    # Add value labels
    labels = [f'{analysis_results["chi2_stat"]:.2f}',
              f'{analysis_results["p_value"]:.4f}',
              f'{analysis_results["position_bias_score"]:.2f}']
    
    for bar, label in zip(bars3, labels):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + max(values) * 0.01,
                label, ha='center', va='bottom')
    
    # 4. Summary text
    ax4.axis('off')
    summary_text = f"""
Model: {model}
Total Valid Responses: {analysis_results['total_responses']:,}
Failed Responses: {analysis_results['failed_responses']}
Overall Accuracy: {analysis_results['overall_accuracy']:.1%}

Position Bias Score: {analysis_results['position_bias_score']:.2f}
Chi-square: {analysis_results['chi2_stat']:.3f}
P-value: {analysis_results['p_value']:.6f}

Conclusion: {'BIAS DETECTED' if analysis_results['p_value'] < 0.05 else 'MINIMAL BIAS'}
    """
    
    ax4.text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center',
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    return fig

print("✓ Visualization functions defined!")

## 12. Run the Full Evaluation

Now let's run the complete evaluation! This cell will:
1. Load the questions
2. Run the model evaluation with permutations
3. Analyze the results
4. Create visualizations
5. Save everything to CSV

**Note:** This may take several minutes depending on your model and number of questions.

In [ ]:
# Run the complete evaluation
print("🚀 Starting complete evaluation...")
print(f"This will process {MAX_QUESTIONS if MAX_QUESTIONS else 'all'} questions with {N_PERMUTATIONS} permutations each.")
print(f"Estimated time: {(MAX_QUESTIONS or 50) * N_PERMUTATIONS * 2 / 60:.1f} minutes (rough estimate)")

# Run evaluation
results_df, output_file = run_evaluation(
    model=MODEL_NAME,
    host=OLLAMA_HOST,
    csv_path=CSV_PATH,
    n_permutations=N_PERMUTATIONS,
    temperature=TEMPERATURE,
    seed=SEED,
    max_questions=MAX_QUESTIONS,
    output_prefix=OUTPUT_PREFIX
)

if results_df is not None:
    print("\n✅ Evaluation completed successfully!")
    
    # Analyze results
    analysis_results = analyze_results(results_df, MODEL_NAME, output_file)
    
    # Create visualizations
    if analysis_results:
        print("\n📊 Creating visualizations...")
        fig = create_visualizations(analysis_results, MODEL_NAME)
        
        # Save the plot
        plot_file = output_file.replace('.csv', '_analysis.png')
        fig.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"📈 Visualization saved to: {plot_file}")
    
    print("\n🎉 All done! Check the results above and the saved files.")
else:
    print("❌ Evaluation failed. Please check the error messages above.")

## 13. Quick Results Summary

View a summary of the key findings:

In [ ]:
# Quick summary of results
if 'results_df' in locals() and results_df is not None:
    print("📋 QUICK SUMMARY")
    print("=" * 50)
    
    valid_responses = results_df[results_df["predicted_answer"].isin(["A", "B", "C", "D"])]
    choice_counts = valid_responses["predicted_answer"].value_counts().reindex(["A", "B", "C", "D"], fill_value=0)
    total = len(valid_responses)
    
    print(f"Model: {MODEL_NAME}")
    print(f"Questions processed: {len(results_df['question_id'].unique())}")
    print(f"Total responses: {len(results_df)}")
    print(f"Valid responses: {total}")
    print(f"Failed responses: {len(results_df) - total}")
    
    print(f"\nChoice Distribution:")
    for letter in ['A', 'B', 'C', 'D']:
        count = choice_counts[letter]
        pct = count / total * 100 if total > 0 else 0
        bar = '█' * int(pct // 2)  # Simple bar visualization
        print(f"  {letter}: {pct:5.1f}% {bar} ({count})")
    
    # Calculate bias
    expected = total / 4
    max_deviation = max(abs(choice_counts[letter] - expected) for letter in ['A', 'B', 'C', 'D'])
    bias_pct = max_deviation / expected * 100
    
    print(f"\nBias Assessment:")
    print(f"  Maximum deviation from expected: {bias_pct:.1f}%")
    if bias_pct > 20:
        print(f"  🚨 HIGH bias detected")
    elif bias_pct > 10:
        print(f"  ⚠️  MODERATE bias detected")
    else:
        print(f"  ✅ LOW bias - distribution is relatively uniform")
        
    print(f"\n💾 Detailed results saved to: {output_file}")
else:
    print("❌ No results available. Please run the evaluation first.")

## 14. Load and Analyze Existing Results (Optional)

If you have existing result files, you can load and analyze them here:

In [ ]:
# Optional: Load existing results for analysis
# Uncomment and modify the filename below to load existing results

# existing_file = "results/positional_bias_qwen2.5_14b-instruct-q8_0.csv"
# 
# if os.path.exists(existing_file):
#     print(f"Loading existing results from: {existing_file}")
#     existing_df = pd.read_csv(existing_file)
#     
#     # Extract model name from filename or dataframe
#     model_name = existing_df['model'].iloc[0] if 'model' in existing_df.columns else "Unknown Model"
#     
#     # Analyze the existing results
#     existing_analysis = analyze_results(existing_df, model_name, existing_file)
#     
#     # Create visualizations
#     if existing_analysis:
#         existing_fig = create_visualizations(existing_analysis, model_name)
#         existing_fig.savefig(existing_file.replace('.csv', '_analysis.png'), dpi=300, bbox_inches='tight')
# else:
#     print(f"File not found: {existing_file}")

print("💡 To load existing results, uncomment and modify the code above with your filename.")

## 15. Compare Multiple Models (Optional)

If you have results from multiple models, you can compare them:

In [ ]:
# Optional: Compare multiple models
# Add your result files here
result_files = [
    # "results/positional_bias_model1.csv",
    # "results/positional_bias_model2.csv",
    # Add more files as needed
]

def compare_models(file_list):
    """Compare bias across multiple models"""
    comparison_data = []
    
    for file_path in file_list:
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            model_name = df['model'].iloc[0] if 'model' in df.columns else os.path.basename(file_path)
            
            valid_responses = df[df["predicted_answer"].isin(["A", "B", "C", "D"])]
            choice_counts = valid_responses["predicted_answer"].value_counts().reindex(["A", "B", "C", "D"], fill_value=0)
            total = len(valid_responses)
            
            if total > 0:
                choice_percentages = choice_counts.values / total * 100
                bias_score = np.std(choice_percentages)
                
                # Chi-square test
                expected = [total / 4] * 4
                chi2_stat, p_value = chisquare(choice_counts.values, f_exp=expected)
                
                comparison_data.append({
                    'Model': model_name,
                    'Total_Responses': total,
                    'A_Percent': choice_percentages[0],
                    'B_Percent': choice_percentages[1],
                    'C_Percent': choice_percentages[2],
                    'D_Percent': choice_percentages[3],
                    'Bias_Score': bias_score,
                    'Chi2_Stat': chi2_stat,
                    'P_Value': p_value,
                    'Has_Bias': p_value < 0.05
                })
    
    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)
        
        print("🔍 MODEL COMPARISON")
        print("=" * 80)
        print(comp_df.to_string(index=False, float_format='%.2f'))
        
        # Create comparison visualization
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Bias scores comparison
        ax1.bar(comp_df['Model'], comp_df['Bias_Score'])
        ax1.set_title('Position Bias Score by Model')
        ax1.set_ylabel('Bias Score (lower = less biased)')
        ax1.tick_params(axis='x', rotation=45)
        
        # Choice distribution heatmap
        choice_data = comp_df[['A_Percent', 'B_Percent', 'C_Percent', 'D_Percent']].values
        im = ax2.imshow(choice_data, cmap='RdYlBu_r', aspect='auto')
        ax2.set_title('Choice Distribution Heatmap')
        ax2.set_xlabel('Answer Choices')
        ax2.set_ylabel('Models')
        ax2.set_xticks(range(4))
        ax2.set_xticklabels(['A', 'B', 'C', 'D'])
        ax2.set_yticks(range(len(comp_df)))
        ax2.set_yticklabels(comp_df['Model'], rotation=0)
        
        # Add colorbar
        plt.colorbar(im, ax=ax2, label='Percentage (%)')
        
        plt.tight_layout()
        plt.show()
        
        return comp_df
    else:
        print("No valid result files found for comparison.")
        return None

if result_files:
    comparison_results = compare_models(result_files)
else:
    print("💡 To compare models, add your result file paths to the 'result_files' list above.")

## Conclusion

This notebook provides a comprehensive evaluation of positional bias in LLM models. The key outputs include:

1. **Choice Distribution**: How often the model picks each position (A, B, C, D)
2. **Statistical Tests**: Chi-square test to detect significant deviations from uniform distribution
3. **Bias Score**: Standard deviation of choice percentages (higher = more biased)
4. **Accuracy Analysis**: Performance broken down by correct answer position
5. **Visualizations**: Charts and graphs showing the bias patterns

### Interpreting Results:

- **No Bias**: All positions chosen ~25% of the time (uniform distribution)
- **Position Bias**: Significant preference for certain positions (A, B, C, or D)
- **Statistical Significance**: P-value < 0.05 indicates significant bias
- **Bias Score**: 
  - < 5: Minimal bias
  - 5-10: Moderate bias  
  - > 10: High bias

### Next Steps:
1. Test with different models to compare bias levels
2. Vary the number of permutations to ensure robust results
3. Try different prompt formats to see if they affect bias
4. Analyze bias patterns across different question types or domains

---

**Files Generated:**
- Results CSV: Contains all raw data and responses
- Analysis PNG: Visualization charts
- This notebook: Complete analysis with visible outputs

Share this notebook to show your complete evaluation process and results! 🎉